In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df=pd.read_csv('loan_approval_data.csv')

In [ ]:
df.head()
df.info()#1000-m(samples),1-o/p,19-features
df.isnull().sum()
df.describe()

In [ ]:
#Handle missing values
categorical_cols=df.select_dtypes(include=["object"]).columns
numerical_cols=df.select_dtypes(include=["float64"]).columns#number in general

In [ ]:
categorical_cols.size+numerical_cols.size

In [ ]:
from sklearn.impute import SimpleImputer
num_imp=SimpleImputer(strategy="mean")
(df[numerical_cols])=num_imp.fit_transform(df[numerical_cols])

In [ ]:
df.head()

In [ ]:
cat_imp=SimpleImputer(strategy="most_frequent")
(df[categorical_cols])=cat_imp.fit_transform(df[categorical_cols])

In [ ]:
df.head()
df.isnull().sum()

In [ ]:
#eda-exploratory data analysis
#how balanced our classes are ?
classes_count=df["Loan_Approved"].value_counts()
plt.pie(classes_count,labels=["No","Yes"],autopct="%1.1f%%")
plt.title("Is loan approved or not?")

In [ ]:
#analyse categories

# gender_count=df["Gender"].value_counts()
# ax=sns.barplot(gender_count)
# ax.bar_label(ax.containers[0])

edu_count=df["Education_Level"].value_counts()
ax=sns.barplot(edu_count)
ax.bar_label(ax.containers[0])

In [ ]:
#analyze income
sns.histplot(
    data=df,
    x="Coapplicant_Income",
    bins=20
)

In [ ]:
#outliers
sns.boxplot(
    data=df,
    x="Loan_Approved",
    y="Applicant_Income"
)

In [ ]:
fig,axes=plt.subplots(2,2)
sns.boxplot (ax=axes[0,0] ,data=df, x="Loan_Approved", y="Applicant_Income")
sns.boxplot (ax=axes[1,0] ,data=df, x="Loan_Approved", y="Credit_Score")
sns.boxplot (ax=axes[0,1] ,data=df, x="Loan_Approved", y="DTI_Ratio")
sns.boxplot (ax=axes[1,1] ,data=df, x="Loan_Approved", y="Savings")

plt.tight_layout()

In [ ]:
#credit score with loan approved
sns.histplot(
    data=df,x="Credit_Score",
    hue="Loan_Approved",#applicant income can also be feature
    bins=20,multiple="dodge"
)

In [ ]:
#encoding 
import pandas as pd
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
le=LabelEncoder()
df["Education_Level"]=le.fit_transform(df["Education_Level"])
df["Loan_Approved"]=le.fit_transform(df["Loan_Approved"])




In [ ]:
df.head()

In [ ]:
cols=["Employment_Status","Marital_Status","Loan_Purpose","Property_Area","Gender","Employer_Category"]

ohe=OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")

encoded = ohe.fit_transform(df[cols])
#ohe.get_feature_names_out(cols)
encoded_df=pd.DataFrame(encoded,columns=ohe.get_feature_names_out(cols),index=df.index)

df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

In [ ]:
#print(df.columns.tolist())# was getting error, used gpt 
ohe.get_feature_names_out(cols)

In [ ]:
df.head()

In [ ]:
#coorelation heatmap: visual rs beet numerical variables in a dataset

nums_cols=df.select_dtypes(include="number")
corr_matrix=nums_cols.corr()
plt.figure(figsize=(15,8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

In [ ]:
corr_matrix

In [ ]:
nums_cols.corr()["Loan_Approved"].sort_values(ascending=False)

In [ ]:
#train-test-split +feature scaling
X=df.drop("Loan_Approved",axis=1)
y=df["Loan_Approved"]

In [ ]:
X.head()

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_test.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
#train n evaluate models
#logistic regression
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score
log_model=LogisticRegression()
log_model.fit(X_train_scaled,y_train)
y_pred=log_model.predict(X_test_scaled)

#evaluation
#precision n recall for 2 issues minimise
print("Precision:",precision_score(y_test,y_pred))
print("Accuracy:",accuracy_score(y_test,y_pred))
print("F1:",f1_score(y_test,y_pred))
print("Recall:",recall_score(y_test,y_pred))
print("Confusion Matrix:",confusion_matrix(y_test,y_pred))


In [ ]:
#knn
from sklearn.neighbors import KNeighborsClassifier
knn_model=KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled,y_train)
y_pred= knn_model.predict(X_test_scaled)

#evaluation
print("KNN Model")
print("Precision : ",precision_score(y_test,y_pred))
print("Accuracy : ",accuracy_score(y_test,y_pred))
print("CM : ",confusion_matrix(y_test,y_pred))
print("Recall : ",recall_score(y_test,y_pred))
print("F1 score : ",f1_score(y_test,y_pred))




In [ ]:
#naive bayes
from sklearn.naive_bayes import GaussianNB

nb_model=GaussianNB()
nb_model.fit(X_train_scaled,y_train)

y_pred=nb_model.predict(X_test_scaled)
#evaluation
print("Naive Bayes Model")
print("Precision : ",precision_score(y_test,y_pred))
print("Accuracy : ",accuracy_score(y_test,y_pred))
print("CM : ",confusion_matrix(y_test,y_pred))
print("Recall : ",recall_score(y_test,y_pred))
print("F1 score : ",f1_score(y_test,y_pred))

In [ ]:
#best model on the basis of precision= Naive bayes
#feature engg.
#add or transform features
df["DTI_Ratio_sq"]=df["DTI_Ratio"]**2
df["Credit_Score_sq"]=df["Credit_Score"]**2

#df["Application_Income_log"]=np.log1p(df["Application_Income"])
x=df.drop(columns=["Loan_Approved","Credit_Score","DTI_Ratio"])
y=df["Loan_Approved"]
#Train_Test_Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
#Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score, recall_score, f1_score

log_model=LogisticRegression()
log_model.fit(X_train_scaled,y_train)
y_pred=log_model.predict(X_test_scaled)

#evaluation
print("Logistic Regression Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))



In [ ]:
# KNN

from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled, y_train)

y_pred = knn_model.predict(X_test_scaled)

# Evaluation
print("KNN Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))

In [ ]:
# Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# Evaluation
print("Naive Bayes Model")
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 score: ", f1_score(y_test, y_pred))
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("CM: ", confusion_matrix(y_test, y_pred))